# 02 — Knowledge Tree Builder

Build and incrementally expand a reusable educational knowledge tree.

This notebook:

1. Loads an existing tree or creates the root tree.
2. Expands a limited number of eligible nodes with the local LLM.
3. Stops expansion at the configured maximum depth.
4. Saves progress back to JSON.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.client import ask_llm
from educational_shorts.prompts import load_prompt
from educational_shorts.schemas import KnowledgeNode
from educational_shorts.tree import (
    count_nodes,
    expand_tree,
    load_tree,
    save_tree,
)

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\hitch\python_files\educational_shorts


## Configuration

`MAX_EXPANSIONS_PER_RUN` controls how many LLM calls this notebook makes each time.

With `MAX_DEPTH = 3`, the root is depth 0 and depth-3 nodes are created but not expanded further.


In [2]:
ROOT_CATEGORY = "Science"
CHILDREN_PER_NODE = 8
MAX_DEPTH = 3
MAX_EXPANSIONS_PER_RUN = 3

TREE_DIRECTORY = PROJECT_ROOT / "data" / "knowledge_tree"
TREE_PATH = TREE_DIRECTORY / f"{ROOT_CATEGORY.lower().replace(' ', '_')}.json"

print(f"Tree path: {TREE_PATH}")

Tree path: c:\Users\hitch\python_files\educational_shorts\data\knowledge_tree\science.json


## Load prompts

In [3]:
root_system_prompt = load_prompt("topic_generation")
expansion_system_prompt = load_prompt("knowledge_expansion")

print("Prompts loaded.")

Prompts loaded.


## Load or create the tree

In [4]:
if TREE_PATH.exists():
    knowledge_tree = load_tree(TREE_PATH)
    print(f"Loaded existing tree from {TREE_PATH}")
else:
    root_user_prompt = f"""
Create a knowledge node for the category "{ROOT_CATEGORY}".

Generate exactly {CHILDREN_PER_NODE} immediate subcategories.

The root node should be named "{ROOT_CATEGORY}".

Each child should have an empty children list.
"""

    knowledge_tree = ask_llm(
        system_prompt=root_system_prompt,
        user_prompt=root_user_prompt,
        schema=KnowledgeNode,
    )

    save_tree(
        tree=knowledge_tree,
        output_path=TREE_PATH,
    )

    print(f"Created and saved a new tree to {TREE_PATH}")

print(f"Current node count: {count_nodes(knowledge_tree)}")

Loaded existing tree from c:\Users\hitch\python_files\educational_shorts\data\knowledge_tree\science.json
Current node count: 57


## Expand the tree

Each expansion selects the next eligible leaf node, asks the LLM for its immediate children, and updates the tree in memory.


In [5]:
nodes_before = count_nodes(knowledge_tree)

expanded_names = expand_tree(
    tree=knowledge_tree,
    system_prompt=expansion_system_prompt,
    max_nodes=MAX_EXPANSIONS_PER_RUN,
    max_depth=MAX_DEPTH,
)

nodes_after = count_nodes(knowledge_tree)

if expanded_names:
    print("Expanded nodes:")
    for name in expanded_names:
        print(f"- {name}")
else:
    print(f"No eligible nodes remain below depth {MAX_DEPTH}.")

print(f"Nodes added: {nodes_after - nodes_before}")
print(f"Total nodes: {nodes_after}")

Expanded nodes:
- Anatomy
- Physiology
- Biochemistry
Nodes added: 24
Total nodes: 81


## Save progress

In [6]:
save_tree(
    tree=knowledge_tree,
    output_path=TREE_PATH,
)

print(f"Saved updated tree to {TREE_PATH}")

Saved updated tree to c:\Users\hitch\python_files\educational_shorts\data\knowledge_tree\science.json


## Optional preview

Use this only when you want to inspect the full JSON tree. Large trees can produce a long output.


In [7]:
# Uncomment to inspect the complete tree.
# print(knowledge_tree.model_dump_json(indent=2))